# Clase 6: Visualización Táctica - El Arte de Guerra con Datos

## Objetivo
Ya sabemos hacer gráficos. Hoy aprenderemos a **contar historias tácticas**. No quiero que me digan 'cuál es el promedio'. Quiero que me digan **por qué perdimos** o **cómo ganar el próximo partido**.

### Los 3 Casos de Estudio:
1. **Baseball:** La Crisis de Eficiencia (Moneyball).
2. **Basketball:** La Trampa de la Media Distancia.
3. **Soccer:** El Delantero Invisible.

---


## 0. Preparación del Entorno (War Room)
Configuramos nuestras herramientas y cargamos la inteligencia disponible.


In [ ]:
# Configuración Robusta
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Detectar Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Verificar Kaggle
if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')) and not os.path.exists('kaggle.json'):
    if IN_COLAB:
        print('Sube tu kaggle.json:')
        files.upload()
    else:
        print('⚠️ Falta kaggle.json')

if os.path.exists('kaggle.json'):
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# Descargar Datos Baseball
if not os.path.exists("baseball_data") and os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    !kaggle datasets download -d open-source-sports/baseball-databank
    !unzip -q baseball-databank.zip -d baseball_data


## Caso 1: La Crisis de Eficiencia (Baseball)

**Situación:** La directiva está furiosa. Hemos gastado $200 Millones en salarios y no estamos en zona de Playoffs. 
**Pregunta:** ¿Quién está haciendo más con menos? ¿Quiénes son los 'pasajeros' de la liga?
**Herramienta:** Scatter Plot (Relación Costo-Beneficio).


In [ ]:
# Carga de Datos Reales
try:
    salaries = pd.read_csv('baseball_data/core/Salaries.csv')
    teams = pd.read_csv('baseball_data/core/Teams.csv')
except FileNotFoundError:
    salaries = pd.read_csv('baseball_data/Salaries.csv')
    teams = pd.read_csv('baseball_data/Teams.csv')

# Preparación: Suma de Salarios por Equipo y Año
team_payroll = salaries.groupby(['yearID', 'teamID'])['salary'].sum().reset_index()
team_performance = pd.merge(team_payroll, teams, on=['yearID', 'teamID'])

# Filtro: Año 2010
data_2010 = team_performance[team_performance['yearID'] == 2010]

# Visualización Táctica
plt.figure(figsize=(12, 8))
sns.scatterplot(data=data_2010, x='salary', y='W', s=100, hue='W', palette='RdYlGn')

# Zona de Peligro (Alto Costo, Pocas Victorias)
plt.axvline(x=data_2010['salary'].mean(), color='gray', linestyle='--')
plt.axhline(y=81, color='gray', linestyle='--') # 81 es .500 de victorias

# Etiquetas Reveladoras
for i in range(len(data_2010)):
    row = data_2010.iloc[i]
    # Etiquetar solo los casos extremos (Outliers)
    if row['salary'] > 1.2e8 or row['W'] > 95 or (row['salary'] < 6e7 and row['W'] > 85):
        plt.text(row['salary']+2e6, row['W'], row['teamID'], weight='bold')

plt.title('Mapa de Eficiencia 2010: ¿Quién gasta mal?')
plt.xlabel('Nómina Total (USD)')
plt.ylabel('Victorias Totales')
plt.grid(True, alpha=0.3)
plt.show()


## Caso 2: La Trampa de la Media Distancia (Basketball)

**Situación:** El Coach cree que nuestro escolta estrella está tomando 'tiros perezosos'.
**Hipótesis:** Está tirando demasiados 'Long 2s' (tiros largos de 2 puntos) que son ineficientes, en lugar de atacar el aro o tirar de 3.
**Herramienta:** Heatmap (Mapa de Calor) de Selección de Tiro.


In [ ]:
# Simulación de Datos: El 'Tirador Perezoso'
np.random.seed(99)
n_shots = 500

# Generamos tiros concentrados en la zona ineficiente (entre la pintura y la linea de 3)
# Coordenadas: Aro en (0,0). Linea de 3 aprox a 23 pies (230 unidades)
r = np.random.normal(180, 40, n_shots) # Radio medio: 18 pies (Long 2s)
theta = np.random.uniform(0, np.pi, n_shots) # Semicírculo

x = r * np.cos(theta)
y = r * np.sin(theta)

shots_df = pd.DataFrame({'x': x, 'y': y})

# Visualización
plt.figure(figsize=(10, 8))
sns.kdeplot(data=shots_df, x='x', y='y', fill=True, cmap='magma', levels=20)

# Dibujar la cancha (simplificada)
plt.xlim(-250, 250)
plt.ylim(-50, 400)
# Aro
plt.gca().add_patch(plt.Circle((0, 0), 7.5, color='orange', fill=False, lw=2))
# Linea de 3 Puntos (aprox)
theta_3pt = np.linspace(0, np.pi, 100)
plt.plot(237.5 * np.cos(theta_3pt), 237.5 * np.sin(theta_3pt), color='white', lw=2, linestyle='--')

plt.title('Diagnóstico de Tiro: La Zona Muerta')
plt.axis('off') # Quitar ejes para parecer pizarra
plt.gca().set_facecolor('black') # Modo oscuro
plt.show()


## Caso 3: El Delantero Invisible (Soccer)

**Situación:** Nuestro delantero centro (#9) se queja de que no le llegan balones.
**La Realidad:** Los datos de tracking sugieren que no se mueve para desmarcarse.
**Herramienta:** Radar Chart Comparativo (Nosotros vs El Estándar de la Liga).


In [ ]:
# Datos: Nuestro Jugador vs Promedio de la Liga
from math import pi

categories = ['Distancia Recorrida', 'Sprints', 'Presión Alta', 'Desmarques', 'Toques en Área']
N = len(categories)

# Valores (Escala 0-100 percentil)
values_our_player = [30, 25, 20, 40, 85] # Estático, solo espera en el área
values_league_avg = [60, 60, 55, 60, 50]

# Cerrar el polígono
values_our_player += values_our_player[:1]
values_league_avg += values_league_avg[:1]
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

plt.figure(figsize=(8, 8))
ax = plt.subplot(111, polar=True)

# Plot Nuestro Jugador
ax.plot(angles, values_our_player, linewidth=2, linestyle='solid', label='Nuestro #9', color='red')
ax.fill(angles, values_our_player, 'red', alpha=0.2)

# Plot Liga
ax.plot(angles, values_league_avg, linewidth=2, linestyle='dashed', label='Promedio Liga', color='blue')

plt.xticks(angles[:-1], categories)
plt.title('Análisis de Actividad: ¿Por qué no recibe el balón?')
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
plt.show()


---
## Tarea: Consultoría Táctica
Ahora tú eres el analista. Resuelve los siguientes misterios.

### Misterio 1: La Maldición del Lado Izquierdo (Basketball)
**Problema:** Fallamos el 75% de los tiros desde la esquina izquierda, pero solo el 40% desde la derecha.
**Tu Misión:** Genera un gráfico que compare la **densidad de tiros** y el **resultado (Made/Missed)** para confirmar si es un problema de ejecución (tiran bien pero fallan) o de selección (tiran forzados).
**Pista:** Filtra por `side='left'` y usa `hue='outcome'`.

### Misterio 2: Rompiendo la Muralla (Soccer)
**Problema:** El próximo rival tiene defensas centrales muy lentos pero fuertes por arriba.
**Tu Misión:** Simula un mapa de los goles que han concedido esta temporada. Si la mayoría son por el centro y rasos, validamos la estrategia de atacar por ahí.
**Pista:** Crea un Scatter plot de 'Goles Concedidos Rival' donde `x` sea la coordenada frontal al arco.


In [ ]:
# Solución Misterio 1: Genera datos sintéticos donde el lado izquierdo tenga muchos 'Missed'
# y grafica un Scatterplot con hue='outcome'.


In [ ]:
# Solución Misterio 2: Genera disparos recibidos concentrados en el centro del área (Zone 14)
# y visualiza con un KDEplot o Scatter.
